# Part 3: Add workplace context from Work IQ

Part 3 at Caldova begins with seeded workplace messages about Summit Dose and `CALD-201`. The messages capture current actions and decisions, while the approved sourcing documents define formal commitments. You need to compare them without treating an email or chat as a contractual record.

**🎯 Mission**
- Preserve Caldova's purchasing policy and both **Indexed Knowledge Sources** from Part 1
- Add a **Work IQ Knowledge Source** for recent Summit Dose workplace context
- Compare current actions and decisions with approved `CALD-201` commitments while keeping the evidence types distinct

## How to read this notebook

Follow **section map > code > Compare with the product**. The orange box marks the current focus. Blue boxes provide context. Section arrows show a learning sequence, not a live deployment status.

Start with the [service roles and full learning path](README.md). Genuine portal screenshots appear only at matching checkpoints. These architecture maps are diagrams, not screenshots. Use the portal paths and actual service/query evidence to inspect your own environment.


## Where you are in the architecture

This is a sibling of the Part 2 KB. It starts with the Part 1 sources and adds Work IQ, not Fabric. Work IQ provides current workplace discussion. Approved commitments still come from sourcing documents. Source access uses the signed-in user's permissions.

[![Part 3 architecture: Work IQ adds permission-scoped workplace context to the Part 1 sources.](../../img/architecture/azure-workiq.png)](../../img/architecture/azure-workiq.png)

**Architecture in brief:** This KB adds permission-scoped Work IQ context to the three Part 1 sources, without Fabric. Workplace discussions complement document evidence but do not establish contractual approval.

**Read the map:** orange marks focused objects, blue arrows show retrieval, purple arrows show model calls, and gray cards are not connected to the focused KB. These are intended connections, not verified deployment states. The checkpoints below read your real service configuration without changing it.

The lab has separate KB variants. Part 1 branches into Part 2 with Fabric and Part 3 with Work IQ. Part 4 combines them. Part 5 reuses the Part 1 sources.

See the [visual field guide](../../docs/visual-guide.md) for portal navigation, expected observations, and the screenshot reference gallery.

## The building blocks

In Parts 1 and 2, your knowledge base queried documents and structured data. Now you'll add a source that connects to your personal workplace context.

- **Work IQ Knowledge Source**: connects to Microsoft 365 (Outlook, Teams, Calendar, OneDrive). The KB queries it at runtime to surface emails, chats, and calendar events relevant to the user's question.
- **User assertion**: This notebook uses `InteractiveBrowserCredential` to sign in the current user and request a Work IQ user assertion for `api://<app-id>/access_as_user`. Azure AI Search exchanges that assertion through a federated credential and calls Work IQ on behalf of the signed-in user, honoring Microsoft 365 permissions.

## Step 1: Set up environment variables

Load the configuration for your Foundry IQ (Azure AI Search) and Azure OpenAI resources. The `.env` file in the project folder has the values needed for this part.

### Section map

![Part 3, step 1: The notebook loads Work IQ identity references and existing Search/model settings.](../../img/architecture/step-configuration.svg)

<details>
<summary>Azure service architecture for this section (target)</summary>

![Official-icon architecture for the workiq phase, with local, Azure, SaaS and identity boundaries.](../../img/architecture/azure-workiq.svg)

**Architecture in brief:** This KB adds permission-scoped Work IQ context to the three Part 1 sources, without Fabric. Workplace discussions complement document evidence but do not establish contractual approval.

This is the phase target, not live status. The process strip above shows the learning sequence. Use the product comparison after the section to verify the actual change.

**Step by step:**

1. Sign in and acquire the delegated user token. Entra prerequisites and the user assertion enable permission-scoped Work IQ retrieval.
2. Create the indexed and Work IQ source definitions, then create a four-source KB with the policy. This branch does not include Fabric.
3. Send the question from the local notebook to the KB over the solid blue connection.
4. Follow the blue connections to policy, document indexes and Work IQ. Workplace context remains in Microsoft 365 and is retrieved with the user's permissions.
5. Follow the purple KB-to-model connection for planning and answer synthesis. Workplace discussion is context, not contractual approval.
6. Inspect Work IQ activity and returned references. Use the notebook SDK/API read-back for Work IQ configuration, not an assumed portal editor.

</details>


In [ ]:
import json
import os

from azure.core.credentials import AzureKeyCredential
from dotenv import load_dotenv

load_dotenv(override=True)


AZURE_SEARCH_SERVICE_ENDPOINT = os.environ["AZURE_SEARCH_SERVICE_ENDPOINT"]
AZURE_SEARCH_ADMIN_KEY = os.environ["AZURE_SEARCH_ADMIN_KEY"]
AZURE_OPENAI_ENDPOINT = os.environ["AZURE_OPENAI_ENDPOINT"]
AZURE_SEARCH_API_VERSION = "2026-08-01-preview"
AZURE_OPENAI_KEY = os.environ["AZURE_OPENAI_KEY"]
AZURE_OPENAI_CHATGPT_DEPLOYMENT = os.environ["AZURE_OPENAI_CHATGPT_DEPLOYMENT"]
AZURE_OPENAI_CHATGPT_MODEL_NAME = os.environ["AZURE_OPENAI_CHATGPT_MODEL_NAME"]
AZURE_TENANT_ID = os.environ["AZURE_TENANT_ID"]
WORK_IQ_ENTRA_APP_ID = os.environ["WORK_IQ_ENTRA_APP_ID"]
WORK_IQ_FEDERATED_CREDENTIAL_ID = os.environ["WORK_IQ_FEDERATED_CREDENTIAL_ID"]
WORK_IQ_ENTRA_TENANT_ID = os.getenv("WORK_IQ_ENTRA_TENANT_ID", AZURE_TENANT_ID)

SUPPLIER_EVIDENCE_INDEX = "supplier-evidence"
SOURCING_DOCUMENTS_INDEX = "sourcing-documents"

search_credential = AzureKeyCredential(AZURE_SEARCH_ADMIN_KEY)

print("Environment variables loaded")

### Open your resources

Run this read-only helper cell to get links built from your local lab configuration. Keep Azure Portal, Fabric, and the notebook side by side. A sign-in page or an access-denied screen is not evidence that a resource is missing.

If the helper import fails, select the lab's `.venv` kernel and use `src/notebooks` as the notebook working directory. Do not copy credentials into the helper or screenshots.

In [ ]:
from lab_visuals import (
    show_activity,
    show_architecture,
    show_file_ingestion,
    show_portal_links,
    show_references,
    verify_knowledge_base,
    verify_sources,
)

show_portal_links(
    AZURE_SEARCH_SERVICE_ENDPOINT,
    search_resource_id=os.getenv("AZURE_SEARCH_RESOURCE_ID", ""),
    app_id=WORK_IQ_ENTRA_APP_ID,
)

### Compare with the product

**Expected after this section:** The notebook loads Work IQ identity references and existing Search/model settings.

**Where to look:** Entra > App registrations > the lab app, and Microsoft Foundry > model deployments.

**What to verify:** Inspect the intended app and tenant privately. No new Entra app or model deployment is created.

This is a navigation and verification guide, not a screenshot or a claim that your environment has completed the step. [Original screenshot availability and redaction rules](../../docs/visual-guide.md#genuine-portal-screenshot-capture).


## Step 2: Login and acquire delegated token

In order for knowledge base retrieval to use your identity with Work IQ, you need an OAuth token for a signed-in user who has access to Microsoft 365 data in this tenant.

Run the cell below to sign in using the credentials in the instructions sidebar. If successful, it acquires a Work IQ user assertion for the `access_as_user` scope on your Work IQ Entra app and saves it for use as `query_work_iq_source_authorization` during retrieval.

### Section map

![Part 3, step 2: An access_as_user assertion is acquired in notebook memory.](../../img/architecture/step-workiq-auth.svg)

<details>
<summary>Azure service architecture for this section (target)</summary>

![Official-icon architecture for the workiq phase, with local, Azure, SaaS and identity boundaries.](../../img/architecture/azure-workiq.svg)

**Architecture in brief:** This KB adds permission-scoped Work IQ context to the three Part 1 sources, without Fabric. Workplace discussions complement document evidence but do not establish contractual approval.

This is the phase target, not live status. The process strip above shows the learning sequence. Use the product comparison after the section to verify the actual change.

**Step by step:**

1. Sign in and acquire the delegated user token. Entra prerequisites and the user assertion enable permission-scoped Work IQ retrieval.
2. Create the indexed and Work IQ source definitions, then create a four-source KB with the policy. This branch does not include Fabric.
3. Send the question from the local notebook to the KB over the solid blue connection.
4. Follow the blue connections to policy, document indexes and Work IQ. Workplace context remains in Microsoft 365 and is retrieved with the user's permissions.
5. Follow the purple KB-to-model connection for planning and answer synthesis. Workplace discussion is context, not contractual approval.
6. Inspect Work IQ activity and returned references. Use the notebook SDK/API read-back for Work IQ configuration, not an assumed portal editor.

</details>


In [ ]:
from azure.identity import InteractiveBrowserCredential

user_credential = InteractiveBrowserCredential(
    client_id=WORK_IQ_ENTRA_APP_ID,
    tenant_id=WORK_IQ_ENTRA_TENANT_ID,
)
user_assertion = user_credential.get_token(
    f"api://{WORK_IQ_ENTRA_APP_ID}/access_as_user"
).token

print("Acquired Work IQ user assertion for logged-in user")

### Compare with the product

**Expected after this section:** An access_as_user assertion is acquired in notebook memory.

**Where to look:** Entra > lab app > Expose an API and API permissions, inspect only.

**What to verify:** The app scope is a prerequisite. Actual Work IQ access is only demonstrated by a successful source call and returned evidence. Never display the assertion.

This is a navigation and verification guide, not a screenshot or a claim that your environment has completed the step. [Original screenshot availability and redaction rules](../../docs/visual-guide.md#genuine-portal-screenshot-capture).


## Step 3: Create the indexed and Work IQ knowledge sources

Reuse the direct policy file source from Part 1, recreate the two indexed source definitions, and add Work IQ:

* `file-docs-knowledge-source`: Caldova's uploaded purchasing policy
* `supplier-evidence-knowledge-source`: Supplier invoices, shipment evidence, production yield records, and quality records
* `sourcing-documents-knowledge-source`: Approved RFP responses, agreements, amendments, and purchase orders
* `workiq-knowledge-source`: Recent workplace messages, email, meetings, and files available through Work IQ

### Section map

![Part 3, step 3: A Work IQ source definition references the preconfigured app and federation.](../../img/architecture/step-workiq-source.svg)

<details>
<summary>Azure service architecture for this section (target)</summary>

![Official-icon architecture for the workiq phase, with local, Azure, SaaS and identity boundaries.](../../img/architecture/azure-workiq.svg)

**Architecture in brief:** This KB adds permission-scoped Work IQ context to the three Part 1 sources, without Fabric. Workplace discussions complement document evidence but do not establish contractual approval.

This is the phase target, not live status. The process strip above shows the learning sequence. Use the product comparison after the section to verify the actual change.

**Step by step:**

1. Sign in and acquire the delegated user token. Entra prerequisites and the user assertion enable permission-scoped Work IQ retrieval.
2. Create the indexed and Work IQ source definitions, then create a four-source KB with the policy. This branch does not include Fabric.
3. Send the question from the local notebook to the KB over the solid blue connection.
4. Follow the blue connections to policy, document indexes and Work IQ. Workplace context remains in Microsoft 365 and is retrieved with the user's permissions.
5. Follow the purple KB-to-model connection for planning and answer synthesis. Workplace discussion is context, not contractual approval.
6. Inspect Work IQ activity and returned references. Use the notebook SDK/API read-back for Work IQ configuration, not an assumed portal editor.

</details>


In [ ]:
from azure.search.documents.indexes import SearchIndexClient
from azure.search.documents.indexes.models import (
    EntraAppAuthentication,
    SearchIndexFieldReference,
    SearchIndexKnowledgeSource,
    SearchIndexKnowledgeSourceParameters,
    WorkIQKnowledgeSource,
    WorkIQKnowledgeSourceParameters,
)

FILE_KNOWLEDGE_SOURCE_NAME = "file-docs-knowledge-source"
SUPPLIER_EVIDENCE_KNOWLEDGE_SOURCE_NAME = "supplier-evidence-knowledge-source"
SOURCING_DOCUMENTS_KNOWLEDGE_SOURCE_NAME = "sourcing-documents-knowledge-source"
WORK_KNOWLEDGE_SOURCE_NAME = "workiq-knowledge-source"
KNOWLEDGE_SOURCE_NAMES = [
    FILE_KNOWLEDGE_SOURCE_NAME,
    SUPPLIER_EVIDENCE_KNOWLEDGE_SOURCE_NAME,
    SOURCING_DOCUMENTS_KNOWLEDGE_SOURCE_NAME,
    WORK_KNOWLEDGE_SOURCE_NAME,
]

index_client = SearchIndexClient(endpoint=AZURE_SEARCH_SERVICE_ENDPOINT, credential=search_credential, api_version=AZURE_SEARCH_API_VERSION)

for knowledge_source_name, index_name, description in [
    (
        SUPPLIER_EVIDENCE_KNOWLEDGE_SOURCE_NAME,
        SUPPLIER_EVIDENCE_INDEX,
        "Caldova supplier invoices, shipment evidence, production yield records, and quality records",
    ),
    (
        SOURCING_DOCUMENTS_KNOWLEDGE_SOURCE_NAME,
        SOURCING_DOCUMENTS_INDEX,
        "Caldova approved RFP responses, agreements, amendments, and purchase orders",
    ),
]:
    knowledge_source = SearchIndexKnowledgeSource(
        name=knowledge_source_name,
        description=description,
        search_index_parameters=SearchIndexKnowledgeSourceParameters(
            search_index_name=index_name,
            source_data_fields=[
                SearchIndexFieldReference(name="blob_path"),
                SearchIndexFieldReference(name="title"),
                SearchIndexFieldReference(name="page_number_from"),
                SearchIndexFieldReference(name="page_number_to"),
            ],
            search_fields=[SearchIndexFieldReference(name="chunk")],
            semantic_configuration_name="semantic-configuration",
        ),
    )
    index_client.create_or_update_knowledge_source(knowledge_source=knowledge_source)

work_knowledge_source = WorkIQKnowledgeSource(
    name=WORK_KNOWLEDGE_SOURCE_NAME,
    description="Caldova workplace context for current Summit Dose actions and decisions",
    work_iq_parameters=WorkIQKnowledgeSourceParameters(
        entra_app_authentication=EntraAppAuthentication(
            application_id=WORK_IQ_ENTRA_APP_ID,
            federated_credential_id=WORK_IQ_FEDERATED_CREDENTIAL_ID,
            tenant_id=WORK_IQ_ENTRA_TENANT_ID,
        )
    ),
)
index_client.create_or_update_knowledge_source(knowledge_source=work_knowledge_source)
print("Knowledge sources created")


### See what you built: source connections

**Portal check:** Work IQ source configuration is currently SDK/API-only, so there is no equivalent Search portal editor to inspect. If permitted, use the Entra link above to inspect **API permissions**, **Expose an API**, and **Certificates & secrets > Federated credentials**. Do not change these preconfigured settings. After querying, open a returned workplace citation.

**Expected:** The source read-back reports `workIQ`. Entra prerequisites alone do not prove user-authorized retrieval works.

**Identity boundary:** the Search service credential manages configuration. Your delegated token or Work IQ assertion authorizes access to protected source data. Never display those tokens.

In [ ]:
verify_sources(index_client, KNOWLEDGE_SOURCE_NAMES)

### Compare with the product

**Expected after this section:** A Work IQ source definition references the preconfigured app and federation.

**Where to look:** Entra > lab app > Federated credentials for prerequisites. Source configuration itself is SDK/API-only in this preview.

**What to verify:** Use the source read-back for actual Search configuration. Do not invent or rely on a portal Work IQ editor.

This is a navigation and verification guide, not a screenshot or a claim that your environment has completed the step. [Original screenshot availability and redaction rules](../../docs/visual-guide.md#genuine-portal-screenshot-capture).


### Find this step in Azure portal

**Open:** Azure portal > your lab Search service > Agentic retrieval > Knowledge sources.

**Look for:** Find `workiq-knowledge-source`. The real portal lists its type as **Other** and displays a notice that this type is not supported in the portal.

[![Original Azure portal view: Knowledge source inventory and portal support notice.](../../img/portal/knowledge-sources.png)](../../img/portal/knowledge-sources.png)

**How to read this:** This is an inventory view, not a Work IQ editor. Do not try to repair the Other label or recreate the source in the portal. Use the notebook's SDK read-back for the actual `workIQ` definition.

*Reference capture: 2026-10-20. Gray blocks redact environment identifiers. Orange outlines are added annotations, not status indicators. This image is not a live check of your resources.*


## Step 4: Create the multi-source Work IQ knowledge base

This knowledge base preserves the purchasing policy file and both indexed evidence sources, then adds Work IQ for current workplace context.

The knowledge base uses `outputMode=answerSynthesis` and `retrievalReasoningEffort=low`. Its retrieval instructions require approved `CALD-201` commitments to come from sourcing documents and current Summit Dose actions or decisions to come from Work IQ.

### Section map

![Part 3, step 4: A separate four-source Work IQ KB connects the three Part 1 sources and Work IQ.](../../img/architecture/step-workiq-source.svg)

<details>
<summary>Azure service architecture for this section (target)</summary>

![Official-icon architecture for the workiq phase, with local, Azure, SaaS and identity boundaries.](../../img/architecture/azure-workiq.svg)

**Architecture in brief:** This KB adds permission-scoped Work IQ context to the three Part 1 sources, without Fabric. Workplace discussions complement document evidence but do not establish contractual approval.

This is the phase target, not live status. The process strip above shows the learning sequence. Use the product comparison after the section to verify the actual change.

**Step by step:**

1. Sign in and acquire the delegated user token. Entra prerequisites and the user assertion enable permission-scoped Work IQ retrieval.
2. Create the indexed and Work IQ source definitions, then create a four-source KB with the policy. This branch does not include Fabric.
3. Send the question from the local notebook to the KB over the solid blue connection.
4. Follow the blue connections to policy, document indexes and Work IQ. Workplace context remains in Microsoft 365 and is retrieved with the user's permissions.
5. Follow the purple KB-to-model connection for planning and answer synthesis. Workplace discussion is context, not contractual approval.
6. Inspect Work IQ activity and returned references. Use the notebook SDK/API read-back for Work IQ configuration, not an assumed portal editor.

</details>


In [ ]:
from azure.search.documents.indexes.models import (
    AzureOpenAIVectorizerParameters,
    KnowledgeBase,
    KnowledgeBaseAzureOpenAIModel,
    KnowledgeSourceReference,
)
from azure.search.documents.knowledgebases.models import (
    KnowledgeRetrievalLowReasoningEffort,
    KnowledgeRetrievalOutputMode,
)

KNOWLEDGE_BASE_NAME = "caldova-workiq-knowledge-base"

aoai_params = AzureOpenAIVectorizerParameters(
    resource_url=AZURE_OPENAI_ENDPOINT,
    deployment_name=AZURE_OPENAI_CHATGPT_DEPLOYMENT,
    model_name=AZURE_OPENAI_CHATGPT_MODEL_NAME,
    api_key=AZURE_OPENAI_KEY,
)

knowledge_base = KnowledgeBase(
    name=KNOWLEDGE_BASE_NAME,
    description="Caldova knowledge base combining purchasing policy, indexed procurement evidence, and workplace context",
    models=[KnowledgeBaseAzureOpenAIModel(azure_open_ai_parameters=aoai_params)],
    knowledge_sources=[KnowledgeSourceReference(name=name) for name in KNOWLEDGE_SOURCE_NAMES],
    retrieval_reasoning_effort=KnowledgeRetrievalLowReasoningEffort(),
    output_mode=KnowledgeRetrievalOutputMode.ANSWER_SYNTHESIS,
    retrieval_instructions="Use Work IQ for current Summit Dose actions and decisions, especially equipment mapping, Piedmont subcontractor approval, and PPQ-01 delivery planning. Use approved sourcing documents for CALD-201 commitments. Clearly distinguish workplace discussion from contractual evidence.",
)

index_client.create_or_update_knowledge_base(knowledge_base)
print("Knowledge base created")

### See what you built: this notebook's KB variant

**What changed:** a new KB connects 4 sources. Earlier KB variants still exist.

**Portal check:** Use the read-back below as the authoritative connection inventory. Work IQ source configuration is not supported in the portal. Do not edit or save this KB through a portal experience that does not expose all its source kinds.

**Expected:** the service returns exactly the source membership shown below with answer synthesis enabled. This is configuration verification, not proof that every source will be called or return evidence for the next question.

In [ ]:
verify_knowledge_base(index_client, KNOWLEDGE_BASE_NAME, KNOWLEDGE_SOURCE_NAMES)

### Compare with the product

**Expected after this section:** A separate four-source Work IQ KB connects the three Part 1 sources and Work IQ.

**Where to look:** The notebook's exact KB membership read-back.

**What to verify:** Four sources, without Fabric. Do not save this KB through a portal editor that omits Work IQ source kinds.

This is a navigation and verification guide, not a screenshot or a claim that your environment has completed the step. [Original screenshot availability and redaction rules](../../docs/visual-guide.md#genuine-portal-screenshot-capture).


### Find this step in Azure portal

**Open:** Azure portal > your lab Search service > Agentic retrieval > Knowledge bases > locate the caldova-workiq-knowledge-base row.

**Look for:** In the **Knowledge sources** column for `caldova-workiq-knowledge-base`, count the policy source, the two indexed sources, and `workiq-knowledge-source (Other)`.

[![Original Azure portal view: Knowledge base inventory and source membership.](../../img/portal/knowledge-bases.png)](../../img/portal/knowledge-bases.png)

**How to read this:** Use this list for orientation only. Keep the notebook's read-back as the configuration check. Do not edit a mixed KB containing Work IQ in a portal that cannot represent that source type.

*Reference capture: 2026-10-20. Gray blocks redact environment identifiers. Orange outlines are added annotations, not status indicators. This image is not a live check of your resources.*


### Concept: Compare context with authority

Use Work IQ for current actions, owners and open decisions, then compare them with approved sourcing records. If a message proposes a new date, identify the proposal and check the commitment rather than claiming that the purchase order changed. Missing workplace access is an evidence gap, not permission to invent an update.

[![Workplace updates and approved sourcing records are compared for owners, deadlines, decisions and conflicts.](../../img/concepts/workplace-versus-commitment.png)](../../img/concepts/workplace-versus-commitment.png)

*Concept illustration, not a portal screenshot or a successful query result.*


## Step 5: Query the knowledge base

Ask Work IQ for current Summit Dose actions and decisions, then compare them with approved `CALD-201` commitments from the sourcing documents.

Workplace messages can describe a technology-transfer milestone, testing-subcontractor evidence, capacity confirmation, or an upcoming supply-review decision. Treat these as current discussion until an approved sourcing document supports them.

> ⏳ Expect to wait 1 to 2 minutes for the response because Work IQ increases retrieval latency. The retrieval request specifies `max_runtime_in_seconds` to increase the time the knowledge base waits for source results.

### Section map

![Part 3, step 5: The query compares current workplace discussion with approved commitments.](../../img/architecture/step-workiq-source.svg)

<details>
<summary>Azure service architecture for this section (target)</summary>

![Official-icon architecture for the workiq phase, with local, Azure, SaaS and identity boundaries.](../../img/architecture/azure-workiq.svg)

**Architecture in brief:** This KB adds permission-scoped Work IQ context to the three Part 1 sources, without Fabric. Workplace discussions complement document evidence but do not establish contractual approval.

This is the phase target, not live status. The process strip above shows the learning sequence. Use the product comparison after the section to verify the actual change.

**Step by step:**

1. Sign in and acquire the delegated user token. Entra prerequisites and the user assertion enable permission-scoped Work IQ retrieval.
2. Create the indexed and Work IQ source definitions, then create a four-source KB with the policy. This branch does not include Fabric.
3. Send the question from the local notebook to the KB over the solid blue connection.
4. Follow the blue connections to policy, document indexes and Work IQ. Workplace context remains in Microsoft 365 and is retrieved with the user's permissions.
5. Follow the purple KB-to-model connection for planning and answer synthesis. Workplace discussion is context, not contractual approval.
6. Inspect Work IQ activity and returned references. Use the notebook SDK/API read-back for Work IQ configuration, not an assumed portal editor.

</details>


In [ ]:
from azure.search.documents.knowledgebases import KnowledgeBaseRetrievalClient
from azure.search.documents.knowledgebases.models import (
    FileKnowledgeSourceParams,
    KnowledgeBaseMessage,
    KnowledgeBaseMessageTextContent,
    KnowledgeBaseRetrievalRequest,
    SearchIndexKnowledgeSourceParams,
    WorkIQKnowledgeSourceParams,
)
from IPython.display import Markdown, display

knowledge_base_client = KnowledgeBaseRetrievalClient(
    endpoint=AZURE_SEARCH_SERVICE_ENDPOINT, knowledge_base_name=KNOWLEDGE_BASE_NAME, credential=search_credential, api_version=AZURE_SEARCH_API_VERSION
)

question = (
    "What current Summit Dose updates affect CALD-201? Identify the open equipment-mapping approval, "
    "the missing Piedmont subcontractor evidence and approval decision, and the PPQ-01 April delivery choice. "
    "Compare each with the approved CALD-201 commitments and identify owners, deadlines, and unresolved decisions."
)

retrieval_request = KnowledgeBaseRetrievalRequest(
    messages=[
        KnowledgeBaseMessage(
            role="user",
            content=[KnowledgeBaseMessageTextContent(text=question)],
        )
    ],
    knowledge_source_params=[
        FileKnowledgeSourceParams(
            knowledge_source_name=FILE_KNOWLEDGE_SOURCE_NAME,
            include_references=True,
            include_reference_source_data=True,
            always_query_source=False,
        ),
        SearchIndexKnowledgeSourceParams(
            knowledge_source_name=SUPPLIER_EVIDENCE_KNOWLEDGE_SOURCE_NAME,
            include_references=True,
            include_reference_source_data=True,
            always_query_source=False,
        ),
        SearchIndexKnowledgeSourceParams(
            knowledge_source_name=SOURCING_DOCUMENTS_KNOWLEDGE_SOURCE_NAME,
            include_references=True,
            include_reference_source_data=True,
            always_query_source=True,
        ),
        WorkIQKnowledgeSourceParams(
            knowledge_source_name=WORK_KNOWLEDGE_SOURCE_NAME,
            include_references=True,
            include_reference_source_data=True,
        ),
    ],
    include_activity=True,
    max_runtime_in_seconds=120,
)

result = knowledge_base_client.retrieve(
    retrieval_request=retrieval_request,
    query_work_iq_source_authorization=user_assertion,
)
if not result.response or not result.response[0].content:
    raise RuntimeError("Knowledge base returned no answer content.")
answer_text = getattr(result.response[0].content[0], "text", None)
if not isinstance(answer_text, str):
    raise TypeError("Knowledge base returned an unsupported answer format.")
display(Markdown(answer_text))

### Review the activity log

For this activity log, you will see a "workIQ" step for the call made to Work IQ, along with "searchIndex" steps for each query sent to a search index.

In [ ]:
activities_json = [activity.as_dict() for activity in result.activity or []]
show_activity(activities_json)

### Review the references

Work IQ references have `type: "workIQ"`. Their `sourceData` contains a list of `parts`. A `text` part holds grounded text from your workplace context, with links to the supporting emails, chats, meetings, or files. A `data` part holds the Work IQ citations: each entry has a `targetLink` to the supporting item and an `isCitedInResponse` flag. Indexed sourcing references have `type: "searchIndex"` and include document citation metadata.

Use these reference types to keep recent Summit Dose discussion separate from approved `CALD-201` commitments.

In [ ]:
references_json = [ref.as_dict() for ref in result.references or []]
show_references(references_json, activities_json, KNOWLEDGE_SOURCE_NAMES)

#### 🔍 Source Hunt

Look at the references printed above. Can you identify:

1. Which Work IQ references support the equipment-mapping action, Piedmont approval status, and PPQ-01 delivery decision?
2. Which sourcing documents establish the approved `CALD-201` commitments for transfer, subcontracting, and delivery?
3. Who owns each current action, what is its deadline, and which decisions remain unresolved?
4. Does the answer clearly label workplace discussion separately from approved commitments?

### Compare with the product

**Expected after this section:** The query compares current workplace discussion with approved commitments.

**Where to look:** The notebook Work IQ references and permitted linked workplace items.

**What to verify:** Identify the owner, deadline, and unresolved decision. Keep discussion and contractual evidence distinct. Do not screenshot personal messages.

This is a navigation and verification guide, not a screenshot or a claim that your environment has completed the step. [Original screenshot availability and redaction rules](../../docs/visual-guide.md#genuine-portal-screenshot-capture).


### Architecture checkpoint

[![Part 3 architecture recap: compare the Work IQ connections with the verified source inventory and actual query evidence.](../../img/architecture/azure-workiq.png)](../../img/architecture/azure-workiq.png)

**Architecture in brief:** This KB adds permission-scoped Work IQ context to the three Part 1 sources, without Fabric. Workplace discussions complement document evidence but do not establish contractual approval.

**Before moving on:** identify what was preconfigured, what this notebook created, and which sources actually returned evidence. The diagram is a learning guide, not a live health report.

## ✅ Mission complete

**What you built:**

* ✓ **Inherited procurement sources**: Caldova's purchasing policy plus supplier evidence and approved sourcing documents remain available.
* ✓ **Work IQ Knowledge Source**: Recent Summit Dose messages provide current actions, decisions, and unresolved workplace context.
* ✓ **Evidence-aware comparison**: The knowledge base compares current discussion with approved `CALD-201` commitments without treating workplace messages as contractual evidence.

➡️ Continue to: [Part 4: Combine Work IQ and Fabric IQ](part4-work-iq-fabric-iq-to-kb.ipynb).